## DSAN 6000 Homework 3A: Allocating Tasks to Parallel Workers with `joblib`

## Overview

You made it to the first DSAN 6000 homework introducing a new coding concept! The goal of this part is for you to gain hands-on experience using **`joblib`** to quickly parallelize an **embarrassingly-parallel** task.

In this case, the problem is one that relates back to Week 1 and Week 2 content on the **OnLine Transaction Processing (OLTP)** mode of data collection and processing: the setup is that a designer spoon brand has set up a new website, where users from across the globe can purchase their latest avant garde silverware using their credit cards. Information from these credit card transactions then **streams into their OLTP database** like you saw in the Week 1 demo!

...However, you've been hired by this designer spoon brand (given your reputation as a DSAN graduate expert) because the company has been experiencing an upsurge in the use of **fraudulent (fake) credit card numbers** being used during checkout! How do we detect fake credit card numbers? In the real world, credit card companies often employ something like the [Luhn Algorithm](https://en.wikipedia.org/wiki/Luhn_algorithm) to choose credit card numbers for customers, whereby only *some* of the 16 digits in the card number are randomly generated: the remaining digits are *computed* as "check" digits, via some mathematical operation applied to the digits that were randomly generated.

For this assignment (since the goal is to show parallel processing speedup, not to teach credit card number verification!), we have made up an alternative version of the Luhn Algorithm scheme:

1.  The first 15 digits of the card number are randomly-generated (with each digit uniformly sampled from $\{0, 1, \ldots, 9\}$)
2.  These 15 digits are then treated as a big 15-digit integer $c$, and we compute the **number of primes in the range $[c - 25, c + 25]$**
3.  We then take this total number of primes found within the range and compute its **remainder modulo 10**. The result is used as the **16th digit** of the credit card number.

Run the following code cell to see an example of this procedure in action:

In [151]:
#| label: cc-num-example
import numpy as np
rng = np.random.default_rng(seed=6001)
from sympy.ntheory import isprime

# Generate the first 15 digits
cc_first15 = rng.integers(low=0, high=9, size=15, endpoint=True)
print(f'First 15 digits: {cc_first15}')

# Interpret these digits as a single integer
cc_first15_int = int(''.join([str(d) for d in cc_first15]))
print(f'As integer: {cc_first15_int}')

# Find primes within range (+/- 25)
range_primes = [int(isprime(n)) for n in range(cc_first15_int - 25, cc_first15_int + 25)]
num_primes = sum(range_primes)
print(f'{num_primes} primes found in range [{cc_first15_int} - 25, {cc_first15_int} + 25]')

# Compute the last digit as this number of primes mod 10
cc_last_digit = num_primes % 10
print(f'Computed check digit: {cc_last_digit}')

# Add the check digit to the end of the generated digits to produce a valid
# credit card number
cc_final = np.append(cc_first15, cc_last_digit)
print(f'Final 16-digit credit card number: {cc_final}')

First 15 digits: [9 8 3 7 2 4 9 3 5 6 2 2 7 0 1]
As integer: 983724935622701
3 primes found in range [983724935622701 - 25, 983724935622701 + 25]
Computed check digit: 3
Final 16-digit credit card number: [9 8 3 7 2 4 9 3 5 6 2 2 7 0 1 3]


Using this scheme, then, the *mathematical* problem of **checking for valid credit card numbers** (in the context of this assignment... not in the real world!) is straightforward: take the first 15 digits of a customer-submitted credit card number and see whether or not the **submitted** 16th digit matches the **check digit** produced by the above process.

The *computational* problem is that, checking for fraudulent card numbers **in serial** significantly slows down the site's transaction-processing throughput... which is why you were hired! So, your task will be to use **parallel processing** to see how/whether you can "push" this OLTP system to process a larger volume of transactions per unit of time, without letting invalid credit card numbers through.

## Part 1: Loading and Preparing Data

### Part 1.1: Use `boto3` to Download the `.parquet` File From S3

Since you already figured out how to use `boto3` to connect to and download files from an S3 bucket in HW2, here we have provided code for you importing `boto3` and setting up the `s3` client object. Your task is to use this client to **download** the file at the following URI:

```
s3://dsan6000-data/transactions_100k.parquet
```

To your EC2 instance, saving it to have the same filename in a `data` subfolder (within the `dsan6000-hw03-parallel-processing` folder). We have already included a `.gitignore` file in the template repo, to ensure that this `.parquet` file doesn't get pushed to GitHub, since it is larger than the allowed invidual file size on GitHub!

Also note that you only need to do this download **one time**: while working on the remainder of this part, you should just be **loading** the already-downloaded `.parquet` file, not re-downloading it from the S3 bucket (since its contents, in this case, are not changing).

In [158]:
#| label: q1.1-init
import boto3
s3 = boto3.client('s3')

In [159]:
#| label: q1.1-response
# Your code here: Download transactions_10m.parquet to the data subfolder
s3.download_file('dsan6000-data', 'cc_transactions_100k.parquet', 'data/cc_transactions_100k.parquet')

### Part 1.2: Loading the Data Into Pandas

In [160]:
#| label: q1.2-init
import pandas as pd
import numpy as np

In [161]:
#| label: q1.2-response
oltp_df = pd.read_parquet("data/cc_transactions_100k.parquet")
oltp_df

,timestamp,customer_id,product_id,amount,cc_num
0,2026-08-20 09:11:41.546837,13927,59,70.30 PLN,3684979005201906
1,2026-08-20 09:11:48.636365,96584,45,47.90 JPY,8692661761705576
2,2026-08-20 09:11:48.713547,85012,70,21.67 ZAR,8079389766740026
3,2026-08-20 09:11:48.925565,96518,36,81.23 JPY,1196239118773592
4,2026-08-20 09:11:55.104031,29571,29,76.49 DKK,0218319760284966
...,...,...,...,...,...
99995,2026-08-20 12:52:02.818899,17760,18,41.60 HKD,9814240704736533
99996,2026-08-20 12:52:03.067919,15957,84,29.57 KRW,1955724747130522
99997,2026-08-20 12:52:03.136630,39324,40,52.32 RON,2539111777898551
99998,2026-08-20 12:52:03.309824,29333,47,53.42 NOK,1223969425480400


## Part 2: Verifying Credit Card Numbers in Serial

Your task in this part is to **implement** the function `verify_cc_num()`, as described at the end of the Overview section above: it should take in a **string**, `cc_num`, containing the 16 digits of a credit card number, and should return the Python boolean value `True` if `cc_num` represents a valid card number, and `False` otherwise.

In [176]:
#| label: Q2-response
from sympy.ntheory import isprime

def verify_cc_num(cc_num: str):
  """
  Your code here: implement the credit card number verification process
  described above, returning the Python boolean value True if the card number is
  valid, and False otherwise.
  """
  pass

# Here we have provided two examples of credit card numbers...

# This first card number is *not* valid, so that your code should return False
print(verify_cc_num('3684979005201906'))
# Whereas this second card number *is* valid, so that your code should return
# True in this case
print(verify_cc_num('8692661761705571'))

None
None


Once you are confident that your `verify_cc_num()` function works, run the following cell to apply it to the `cc_num` column of the `oltp_df` DataFrame!

Note that, to make the comparison with parallel processing as fair as possible, this code extracts the `cc_num` column from the full `DataFrame`, using `to_list()` function available on Pandas `Series` objects, and stores these extracted values in a list named `cc_nums`. Them the parallel processing you'll do in the next part will operated on this same list.

Also note that we've added a "wrapper" around our loop, by importing the `tqdm()` function from the `tqdm` library and applying this function to the collection we're iterating over (`cc_nums`). This just adds a nice progress bar to the output cell, which helps to keep track of how quickly the serial code is working. However, note that (for all the reasons discussed in class!) this progress bar "wrapper" approach should **not** be used for the parallel code in the next section (since it will add tons of overhead, requiring each worker to "report back to" `tqdm` after each bit of work). To track the progress of a parallelized job, as you'll see, `joblib` provides its own reporting mechanism.

In [ ]:
#| label: Q2-serial-timing
import time
disp_time = lambda start, end: print('{:.4f} s'.format(end - start))
from tqdm import tqdm
cc_nums = oltp_df['cc_num'].to_list()
print(f'Checking {len(cc_nums)} submitted credit card numbers')

serial_start = time.time()
valid_ccs = [verify_cc_num(num) for num in tqdm(cc_nums)]
serial_end = time.time()
disp_time(serial_start, serial_end)

total_invalid = len(cc_nums) - sum(valid_ccs)
print(f'{total_invalid} fraudulent card numbers detected!')

Congratulations! If you implemented `verify_cc_num()` correctly, you should have detected many many fraudulent credit card submissions from users, saving your company millions of dollars! However, it turns out that the amount of time that the **non-fraudulent** customers now have to wait (for their card numbers to be validated) is leading many of them to abandon your site for a competitor's site 😭 Thus, in the next part you will **parallelize** this validation process, to push back against this cause of customer attrition!

## Part 3: Verifying Credit Card Numbers in Parallel

First things first, let's import `joblib` and have it detect how many **cores** we have on this EC2 instance:

In [175]:
#| label: Q3-init-joblib
import joblib
joblib.cpu_count()

2

As expected, since we're using AWS's `m3.large` setup, we have **two cores** total to work with. In the following cell, we have provided code to initialize a parallel pipeline (called `parallel_runner`) using `joblib`.

When using `joblib`'s `Parallel()` constructor to create a parallel pipeline, you *can* manually tell it to use 2 jobs (or just, however many cores you may have on your machine). However, since it has this ability to *detect* the number of cores on a machine, here we instead use the argument `n_jobs=-1`, which tells `joblib` to use `joblib.cpu_count()`. This is almost always what you would want to do, *unless* there are other tasks you'd like to run on your instance, since `n_jobs=-1` will take up *all* processing power across *all* cores of the computer!

The only other argument to be aware of here is `batch_size='auto'`: this is a lot like the **hyperparameters** you have seen in previous classes, where you could **tune** this to try and optimize time savings. You can provide some **integer** $n$ for this argument, rather than `auto`, and `joblib` will accumulate a batch of $n$ inputs to send to each worker, rather than sending the inputs one at a time. Since (as we talked about a bit in lecture) there is an **overhead cost** involved when sending tasks out to workers, this batching ability can speed things up immensely. However, if you have some extremely computationally-intensive task, you may want to *avoid* batching altogether, to ensure that each worker is assigned exactly *one* task at a given time. In those cases, you can instead use `batch_size=1`.

So, your job is to **complete the code in the following cell**, by drawing on the code shown in class to **apply** the `parallel_runner` object to a **"frozen" version** of the `verify_cc_num` function you wrote above! Remember: you do **not** want to actually **call** the `verify_cc_num` function here! Instead, you should use `joblib`'s `delayed()` function to **"freeze"** the code in `verify_cc_num`, so that it can send a *copy* of this code out to each worker.

Unlike in Part 2, here we are combining the complete-this-code part of the cell with the time-computation task. So, your job here is just to fill in the portion of the code *after* the "timer" starts (the first call to `time.time()`) and *before* it ends (the second call to `time.time()`). In the final part of HW3A below, you will compare this runtime with the serial runtime you achieved above!

In [ ]:
#| label: Q3-response
parallel_runner = joblib.Parallel(n_jobs=-1, batch_size='auto', verbose=True)
par_start = time.time()
# Your code here: replace the None, using parallel_runner to verify the credit
# card numbers in the cc_nums list in parallel

valid_ccs_parallel = None

# (End of your code)
par_end = time.time()
disp_time(par_start, par_end)

total_invalid = len(cc_nums) - sum(valid_ccs_parallel)
print(f'{total_invalid} fraudulent card numbers detected!')

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 7164 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done 83964 tasks      | elapsed:   16.0s


18.9203 s


[Parallel(n_jobs=-1)]: Done 100000 out of 100000 | elapsed:   18.9s finished


## Part 4: Interpreting the Parallelization Overhead

You made it to the end of HW3A! Before moving to HW3B, however, please take note of the difference in total time required between the **serial** and **parallel** approaches, and answer the following quick questions based on that difference:

### Question 4.1

When you carried out the data-processing on **two** cores rather than one, did you achieve a speedup?

In [127]:
#| label: Q4.1-response
q4_1_response = "" # Replace with "Yes" or "No"

### Question 4.2

If you did achieve a speedup, was it *exactly* twice as fast, *more than* twice as fast, or *less than* twice as fast?

*(If you didn't achieve a speedup, just put `"NA"` as your response here)*

In [ ]:
#| label: Q4.2-response
q4_2_response = "" # Replace with "Exactly", "More", "Less", or "NA"

### Question 4.3

If you did achieve a speedup, here compute the exact *speedup amount*, the ratio of the time required to solve the problem in **serial** to the time required to solve the problem in **parallel**

*(Knowing that all parallelization requires some overhead means that this will typically be **less than** the number of cores you have. Then, Amdahl's Law tells us that, even as we add more and more cores, we will eventually hit an asymptotic maximum possible speedup)*

In [128]:
#| label: Q4.3-response
q4_3_response = None # Replace with speedup calculation